In [1]:
!pip install langchain chromadb faiss-cpu tiktoken langchain_huggingface langchain-community wikipedia

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   - -------------------------------------- 0.8/18.9 MB 7.6 MB/s eta 0:00:03
   ------ --------------------------------- 2.9/18.9 MB 8.9 MB/s eta 0:00:02
   ----------- ---------------------------- 5.2/18.9 MB 10.0 MB/s eta 0:00:02
   ---------------- ----------------------- 7.6/18.9 MB 10.5 MB/s eta 0:00:02
   --------------------- ------------------ 10.2/18.9 MB 10.8 MB/s eta 0:00:01
   --------------------------- ------------ 12.8/18.9 MB 11.0 MB/s eta 0:00:01
   ------------------------------- -------- 14.7/18.9 MB 10.8 MB/s eta 0:00:01
   ------------------------------------ --- 17.3/18.9


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_community.retrievers import WikipediaRetriever

In [3]:
retriever = WikipediaRetriever(top_k_results=2, lang="en")

In [4]:
query = 'The exploitation of British from India in the perspective of an american' ,
docs = retriever.invoke(query)


In [5]:
docs

[Document(metadata={'title': 'Exploitation of labour', 'summary': "Exploitation is a concept defined as, in its broadest sense, one agent taking unfair advantage of another agent. When applying this to labour (or labor), it denotes an unjust social relationship based on an asymmetry of power or unequal exchange of value between workers and their employers. When speaking about exploitation, there is a direct affiliation with consumption in social theory and traditionally this would label exploitation as unfairly taking advantage of another person because of their vulnerable position, giving the exploiter the power.\nKarl Marx's theory of exploitation has been described in the Stanford Encyclopedia of Philosophy as the most influential theory of exploitation. Marx described exploitation as the theft of economic power in all class-based societies, including capitalism, through the working class (or the proletariat, as Marx called them) being forced to sell their labour. The two main persp

In [8]:
# Print retrieved content
for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(f"Content:\n{doc.page_content}...")  # truncate for display


--- Result 1 ---
Content:
India–Iran relations are the bilateral relationship between the Republic of India and the Islamic Republic of Iran. Independent India and Iran established diplomatic relations on 15 March 1950.
Contact between both ancient Persia and ancient India date to ancient times, and can be seen through the diffusion of Persian culture among Islamic culture in much of South Asia; furthermore, around 15% of the Muslims in India are Shia, a group Iran considers itself to represent on the world stage. Outside the Islamic community, the impact of Persian culture has primarily been in Northwest India.
During much of the Cold War, relations between India and the erstwhile Imperial State of Iran suffered due to their differing political interests: India endorsed a non-aligned position but fostered strong links with the Soviet Union, while Iran was an open member of the Western Bloc and enjoyed close ties with the United States. While India did not welcome the 1979 Islamic Rev

In [10]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

In [11]:
# Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [13]:
embedding_model = HuggingFaceEmbeddings()

vectorstore = Chroma.from_documents(
    documents = documents,
    embedding = embedding_model,
    collection_name = "my_collection"

)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8069.87it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
# Step 4: Convert vectorstore into a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [15]:
query = "What is Chroma used for?"
results = retriever.invoke(query)

In [16]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
Embeddings convert text into high-dimensional vectors.


In [17]:
results = vectorstore.similarity_search(query, k=2)

In [18]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
Embeddings convert text into high-dimensional vectors.


In [19]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [20]:
from langchain_community.vectorstores import FAISS

# Initialize OpenAI embeddings
embedding_model = HuggingFaceEmbeddings()

# Step 2: Create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7097.87it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [21]:
# Enable MMR in the retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",                   # <-- This enables MMR
    search_kwargs={"k": 3, "lambda_mult": 0.5}  # k = top results, lambda_mult = relevance-diversity balance
)

In [22]:
query = "What is langchain?"
results = retriever.invoke(query)

In [23]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
LangChain is used to build LLM based applications.

--- Result 2 ---
Embeddings are vector representations of text.

--- Result 3 ---
LangChain supports Chroma, FAISS, Pinecone, and more.
